# Round 12 — Reference consistency and reciprocal evidence

Two new feature families; 48 primary columns. Fixed classifier and cached anchor. This is exploratory development, not a Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/reference_consistency_features.json").is_file())
from scripts.run_reference_consistency_features import figures, write_dashboard
SPEC = json.loads((ROOT / "configs/reference_consistency_features.json").read_text())
RESULT = json.loads((ROOT / "reports/reference_consistency_features/results.json").read_text())
print("Registered primary:", SPEC["primary"])
print("New classifier fits:", RESULT["new_fits"])
print("Prior readouts reused:", RESULT["reused_prior_controls"])
CHARTS = figures(RESULT)

Registered primary: consistency_all
New classifier fits: 12
Prior readouts reused: 10


## 1. Hypothesis and unchanged anchor

Does consistency with other labeled references make support evidence more useful without deleting rare or disputed examples?

24 ordinary label-neighborhood consistency features and 24 reciprocal-neighbor consistency features. The context anchor was selected after Round 9 and was not promoted.

In [2]:
display(pd.DataFrame(RESULT["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
7,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367
8,0,"No Advertising: Spam, referral links, unsolici...",uniform_all,0.703246,0.247805,0.821082
9,1,No legal advice: Do not offer or request legal...,uniform_all,0.754574,0.233871,0.817539


## 2. Conditional uncertainty

Per-policy results matter. Intervals cover registered within-round contrasts, not all project-wide adaptive decisions.

In [3]:
display(pd.DataFrame(RESULT["comparisons"]))
CHARTS[1].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,agreement_features,context_evidence,agreement_features vs context anchor,-0.001427,-0.020136,0.017281
1,reciprocity_features,context_evidence,reciprocity_features vs context anchor,-0.002379,-0.021087,0.016330
2,consistency_all,context_evidence,consistency_all vs context anchor,-0.002659,-0.021368,0.016049
3,quality_null_all,context_evidence,quality_null_all vs context anchor,-0.005588,-0.024297,0.013121
4,geometry_only_all,context_evidence,geometry_only_all vs context anchor,-0.010203,-0.028912,0.008505
5,label_null_all,context_evidence,label_null_all vs context anchor,-0.011391,-0.030099,0.007318
6,consistency_all,qwen_raw,Primary vs qwen_raw,0.006704,-0.012004,0.025413
7,consistency_all,frozen_basic,Primary vs frozen_basic,0.003530,-0.015179,0.022239
8,consistency_all,uniform_all,Primary vs uniform_all,-0.002313,-0.021021,0.016396
9,consistency_all,quality_null_all,Primary vs quality_null_all,0.002929,-0.015780,0.021637


## 3. Mechanism controls and family ablations

quality_null_all, geometry_only_all, label_null_all. No secondary winner silently replaces the primary.

In [4]:
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

## 4. Coverage and reference diagnostics

Disagreement is not a label error. No label is edited, removed, or repaired by this experiment.

In [5]:
display(pd.DataFrame(RESULT["diagnostics"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,inner_fold,self_overlap,mode,class,reference_rows,mean_quality,minimum_quality,maximum_quality,labels_changed,query_used_for_quality,neighbors_per_reference,mean_neighbor_agreement,mean_reciprocal_agreement,mean_reciprocal_degree,self_neighbors
0,0,0,0,observed,0,250,0.780808,0.400000,0.916667,False,False,7,0.722355,0.671459,3.274463,0
1,0,0,0,observed,1,169,0.760656,0.333333,0.916667,False,False,7,0.722355,0.671459,3.274463,0
2,0,0,0,quality_null,0,250,0.780808,0.400000,0.916667,False,False,7,0.722355,0.671459,3.274463,0
3,0,0,0,quality_null,1,169,0.760656,0.333333,0.916667,False,False,7,0.722355,0.671459,3.274463,0
4,0,0,0,geometry_only,0,250,0.598000,0.333333,0.916667,False,False,7,0.722355,0.671459,3.274463,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1,outer_query,0,quality_null,1,218,0.797422,0.333333,0.916667,False,False,7,0.736794,0.674707,3.524590,0
60,1,outer_query,0,geometry_only,0,148,0.648649,0.333333,0.916667,False,False,7,0.736794,0.674707,3.524590,0
61,1,outer_query,0,geometry_only,1,218,0.612385,0.333333,0.916667,False,False,7,0.736794,0.674707,3.524590,0
62,1,outer_query,0,label_null,0,148,0.571886,0.333333,0.850000,False,False,7,0.511840,0.499869,3.524590,0


## 5. Fitted associations

Coefficients are not causal effects. Use matched additions/removals to judge evidence.

In [6]:
CHARTS[6].show(renderer="plotly_mimetype")

## 6. Probability quality and metrics

AUC, per-policy macro AUC, and ranked pooled AUC are distinct. Brier/log loss track probability quality. None is a new hidden Kaggle score.

In [7]:
display(pd.DataFrame(RESULT["pooled_metrics"]))
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,agreement_features,0.727830,0.734316,0.738909
6,reciprocity_features,0.726878,0.732264,0.738835
7,consistency_all,0.726598,0.730519,0.736879
8,quality_null_all,0.723669,0.709782,0.738809
9,geometry_only_all,0.719054,0.724195,0.730477


## 7. Fixed decision and limitations

Primary: `consistency_all`. Both companion designs are independent. The inherited training Qwen answer margin remains in-sample even though these new features are cross-fitted.

Research: https://research.google/pubs/confident-learning-estimating-uncertainty-in-dataset-labels/

In [8]:
print("Decision:", RESULT["decision"])
for item in RESULT["primary_requirements"]:
    print(item["reference"], item["per_policy_delta"], "passed:", item["passed"])
for limitation in RESULT["limitations"]:
    print(limitation)
print("Interactive dashboard:", write_dashboard(ROOT, RESULT))

Decision: DO_NOT_PROMOTE_PRIMARY
qwen_raw [0.025335820895522443, -0.011927359542865945] passed: False
frozen_basic [0.011492537313432805, -0.004432398583197994] passed: False
context_evidence [0.0011194029850747356, -0.006438230171620818] passed: False
uniform_all [0.0013432835820896827, -0.005968572043599796] passed: False
quality_null_all [0.013283582089552382, -0.00742646914933176] passed: False
geometry_only_all [0.01000000000000012, 0.005087963053560518] passed: False
label_null_all [-0.002537313432835697, 0.019999608618226583] passed: False
Exploratory adaptive development on the same 881 queries, not a fresh holdout or Kaggle score.
The context_evidence anchor is held fixed, selected after Round 9, and was never promoted.
New reference features are text-group cross-fitted. The inherited adapted answer score is in-sample on support labels.
Cross-fitting the new features cannot repair that inherited answer-score limitation.
Simultaneous intervals cover this round, not the full ada

Interactive dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/reference_consistency_features/dashboard.html
